# Entrega 2 proyecto aprendizaje de máquina

Pedro Pablo Sanín Trujillo (202221527)


## 1 - Carga de datos


In [1]:
# Importar la librerías del proyecto
import pandas as pd
import tensorflow as tf
from keras.layers import TextVectorization
from sklearn.model_selection import train_test_split

In [2]:
# A continuación, se cargan los conjuntos de datos de entrenamiento y evaluación

path_train = "./datos/train.csv"
path_eval = "./datos/eval.csv"


dataset_train_original = pd.read_csv(path_train)
dataset_eval_original = pd.read_csv(path_eval)

dataset_train = dataset_train_original.copy()
dataset_eval = dataset_eval_original.copy()

In [3]:
print(
    f"Número de filas y columnas del conjunto de entrenamiento: {dataset_train.shape}"
)

dataset_train.head(10)

Número de filas y columnas del conjunto de entrenamiento: (20466, 3)


,id,text,label
0,0,Lo pedí un martes por internet y llegó al terc...,negativo
1,1,Les cuento como me fue. Compre esta tablet a m...,positivo
2,2,Lo compré hace como dos meses en una tienda de...,neutral
3,3,Le routeur capte très bien dans toute la maiso...,negativo
4,4,"I got it a while back on sale online, it arriv...",neutral
5,5,Comprei esse teclado mês passado pra trabalhar...,positivo
6,6,Comprei há umas duas semanas pela internet e c...,negativo
7,7,"La compre a mediados de año por internet, tard...",positivo
8,8,Les dejo un comentario rápido. Lo pedí a media...,negativo
9,9,The color accuracy on this monitor is genuinel...,positivo


Vemos que el dataset contiene 20466 filas, cada una con un id, la reseña correspondiente y un label que indica que si la reseña es positiva o negativa. Revisando manualmente algunas entradas, confirmamos que los únicos idiomas presentes en las reseñas son español, inglés, portugués, francés y una mezcla de español e inglés. Esto es consistente con lo indicado en el enunciado del proyecto. Preliminarmente, no separaremos los modelos reseñas por idioma pues esto reduce sustancialmente el tamaño de cada conjunto de datos. Sin embargo, si vale la pena identificar el idioma para tener una división train-test balanceada.

Además nuestra métrica de entrenamiento será el accuracy, de acuerdo con la competencia. A continuación, intentaremos identificar el idioma de cada una de nuestras reseñas.


## 2 - Identificando el idioma de la reseña

Para esto, descargaremos las 50 palabras más comunes en cada uno de los idiomas. Por la ley de Zipf, si las listas de las 50 palabras más comunes son distintas, una de las palabras aparecerá en la lista y esto nos servirá para determinar el idioma de la reseña. Asimismo, las palabras más comunes en cada idioma son artículos y generalmente tienen el mismo significado.

Así, el criterio para determinar el idioma de una reseña será el siguiente:

1. El primer y segundo idioma candidato serán los primeros dos conjuntos de palabras que tengan más coincidencia con las palabras de la reseña.

2. Con las palabras que pertenecen al conjunto más común,


In [4]:
import sys
import unicodedata
from collections import Counter

# Tabla de traducción que reemplaza por un espacio todo carácter Unicode de puntuación
# (categorías 'P*'), lo que incluye ¿ ¡ « » “ ” … además de la puntuación ASCII.
puntua = {
    i: " "
    for i in range(sys.maxunicode + 1)
    if unicodedata.category(chr(i)).startswith("P")
}

# Palabras de la lista de inglés que también son muy comunes en español ("lo he usado"),
# por lo que no cuentan como evidencia de inglés dentro de una reseña en español.
AMBIGUAS_EN = {"he"}


def procesar_resena(text):
    """Pasa la reseña a minúsculas, elimina la puntuación y la separa en palabras."""
    text = text.lower()
    text = text.translate(puntua)
    return text.split()


def palabras_exclusivas(palabras_por_idioma):
    """Para cada idioma, las palabras comunes que no aparecen en la lista de ningún otro idioma."""
    return {
        idioma: palabras
        - set().union(
            *(otras for otro, otras in palabras_por_idioma.items() if otro != idioma)
        )
        for idioma, palabras in palabras_por_idioma.items()
    }


def identificar_idioma(resena, palabras_por_idioma, exclusivas):
    """Identifica el idioma de una reseña: 'es', 'en', 'fr', 'pt_br', 'es_en' (bilingüe) o None.

    1. El idioma principal es el que tiene más ocurrencias de sus palabras exclusivas. Si la reseña
       no tiene ninguna palabra exclusiva, se usan todas las palabras comunes (incluidas las compartidas).
    2. Si el idioma principal es español o inglés y la reseña contiene al menos una palabra exclusiva
       del otro idioma, la reseña se considera bilingüe ('es_en').
    """
    conteo = Counter(procesar_resena(resena))

    def puntajes(listas):
        return {
            idioma: sum(n for palabra, n in conteo.items() if palabra in palabras)
            for idioma, palabras in listas.items()
        }

    puntaje = puntajes(exclusivas)
    if max(puntaje.values()) == 0:
        puntaje = puntajes(palabras_por_idioma)
        if max(puntaje.values()) == 0:
            return None
    principal = max(puntaje, key=lambda k: puntaje[k])

    hay_es = any(palabra in exclusivas["es"] for palabra in conteo)
    hay_en = any(palabra in exclusivas["en"] - AMBIGUAS_EN for palabra in conteo)
    if principal in ("es", "en") and hay_es and hay_en:
        return "es_en"
    return principal

In [5]:
# Cargamos las 50 palabras más comunes por idioma y les aplicamos la misma limpieza que a las reseñas,
# para que entradas como "c'", "l'" o "'s" queden como "c", "l" y "s" y puedan coincidir con las reseñas.
path_palabras = "./datos/50_palabras_mas_comunes/50_palabras_mas_comunes.csv"

palabras_comunes = pd.read_csv(path_palabras)
palabras_comunes["palabra"] = palabras_comunes["palabra"].map(
    lambda p: procesar_resena(p)[0]
)

# Conjunto de palabras por idioma, y el subconjunto de palabras exclusivas de cada idioma
palabras_por_idioma = palabras_comunes.groupby("idioma")["palabra"].apply(set).to_dict()
exclusivas = palabras_exclusivas(palabras_por_idioma)

palabras_comunes.head()

,idioma,rank,palabra,frecuencia
0,en,1,you,28787591
1,en,2,i,27086011
2,en,3,the,22761659
3,en,4,to,17099834
4,en,5,a,14484562


In [6]:
print(
    identificar_idioma("Esta es una reseña de prueba", palabras_por_idioma, exclusivas)
)
print(identificar_idioma("This is a test review", palabras_por_idioma, exclusivas))
print(
    identificar_idioma("Ceci est une critique de test", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma("Esta é uma crítica de teste", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma(
        "Esta es una reseña de prueba en inglés: This is a test review",
        palabras_por_idioma,
        exclusivas,
    )
)
print(
    identificar_idioma(
        "Esta es una reseña de prueba en francés: Ceci est une critique de test",
        palabras_por_idioma,
        exclusivas,
    )
)

es
en
fr
pt_br
es_en
es


In [7]:
# Identificamos el idioma de cada reseña de entrenamiento y evaluación
dataset_train["idioma"] = dataset_train["text"].map(
    lambda t: identificar_idioma(t, palabras_por_idioma, exclusivas)
)
dataset_eval["idioma"] = dataset_eval["text"].map(
    lambda t: identificar_idioma(t, palabras_por_idioma, exclusivas)
)

dataset_train["idioma"].value_counts(dropna=False, normalize=True)

idioma
es       0.652497
en       0.114678
pt_br    0.114043
fr       0.102511
es_en    0.016271
Name: proportion, dtype: float64

In [8]:
dataset_train.head()

,id,text,label,idioma
0,0,Lo pedí un martes por internet y llegó al terc...,negativo,es
1,1,Les cuento como me fue. Compre esta tablet a m...,positivo,es
2,2,Lo compré hace como dos meses en una tienda de...,neutral,es
3,3,Le routeur capte très bien dans toute la maiso...,negativo,fr
4,4,"I got it a while back on sale online, it arriv...",neutral,en


Ahora, dividimos el conjunto de datos en prueba y entrenamiento estratificado para incluir una proporción representativa de cada idioma y de cada etiqueta. Para esto, usaremos `sklearn.train_test_split`.


In [9]:
# Codificamos las etiquetas de como enteros
etiquetas = sorted(
    dataset_train["label"].unique()
)  # ['negativo', 'neutral', 'positivo']
etiqueta_a_id = {e: i for i, e in enumerate(etiquetas)}


# División 90/10 estratificada por idioma
train_df, test_df = train_test_split(
    dataset_train,
    test_size=0.1,
    random_state=42,
    stratify=dataset_train["idioma"],
)


def dataset_a_tf(df, shuffle):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["text"].values, df["label"].map(etiqueta_a_id).values)
    )
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)


train_dataset = dataset_a_tf(train_df, shuffle=True)
test_dataset = dataset_a_tf(test_df, shuffle=False)

## 2 - Red neuronal

In [10]:
encoder_mlp = TextVectorization(output_mode="tf_idf")
encoder_mlp.adapt(train_df["text"])

In [11]:
model_mlp = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(), dtype=tf.string),
        encoder_mlp,
    
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3),
    ]
)

model_mlp.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 19025)          │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │       608,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 608,931 (2.32 MB)

 Trainable params: 608,931 (2.32 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model_mlp.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

Ahora para entrenar el modelo, usaremos ``tf.keras.callbacks.EarlyStopping``. Esto nos permite parar el entrenamiento cuando maximicemos la métrica objetivo, y restaurar los mejores parámetros en caso de que la métrica objetivo baje luego de la iteración.

In [13]:
# En lugar de fijar el número de épocas, entrenamos hasta un máximo de 50 y detenemos el
# entrenamiento cuando el accuracy de validación deja de mejorar durante 3 épocas seguidas.
# restore_best_weights=True deja el modelo con los pesos de la mejor época.
def entrenar_modelo_con_parada_temprana(model, train_dataset, test_dataset, max_epochs=50, patience=3): 
    parada_temprana = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=patience,
    restore_best_weights=True,
)

    history = model.fit(
    train_dataset,
    epochs=max_epochs,
    validation_data=test_dataset,
    callbacks=[parada_temprana],
)

    mejor_epoca = max(
    range(len(history.history["val_accuracy"])),
    key=lambda i: history.history["val_accuracy"][i],
)
    print(
    f"Mejor época: {mejor_epoca + 1} "
    f"(val_accuracy = {history.history['val_accuracy'][mejor_epoca]:.4f})"
)

    return history


In [14]:
entrenar_modelo_con_parada_temprana(model_mlp, train_dataset, test_dataset)

Epoch 1/50


/Users/pedropablosanintrujillo/GitHub/proyecto-ml/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6479 - loss: 0.5863 - val_accuracy: 0.6659 - val_loss: 0.5345
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7963 - loss: 0.4017 - val_accuracy: 0.6771 - val_loss: 0.5657
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8727 - loss: 0.2720 - val_accuracy: 0.6820 - val_loss: 0.6835
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9275 - loss: 0.1759 - val_accuracy: 0.6825 - val_loss: 0.7987
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9594 - loss: 0.1134 - val_accuracy: 0.6839 - val_loss: 0.9167
Epoch 6/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9770 - loss: 0.0753 - val_accuracy: 0.6864 - val_loss: 1.0311
Epoch 7/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9864 - loss: 0.0478 - val_accuracy: 0.6844 - val_loss: 1.1886
Epoch 8/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9907 - loss: 0.0344 - val_accuracy: 0.6864 - val_

### 3.1 - Búsqueda de hiperparámetros 

## 3 - Red neuronal recurrente básica


Inicialmente, usaremos una red neuronal recurrente para clasificar las reseñas. Inicialmente, veremos el desempeño de una RNN simple, donde cada neurona tiene información sobre las operaciones que ha hecho en pasos anteriores. Las redes neuronales recurrentes funcionan bien al procesar lenguaje natural pues el lenguaje tiene una naturaleza secuencial.

Usaremos la implementación de RNNs dada por `tf.keras.layers.SimpleRNN`. Para que el texto pueda pasar exitosamente por una red neuronal, debemos vectorizarlo. Para esto, usaremos una capa inicial de la red neuronal dada por `tf.keras.layers.TextVectorization`. Este algoritmo de vectorización combina varias formas de vectorizar frases como _tf_idf_, _count_ (bag of words) y soporta ngramas.


In [15]:
# Para una red recurrente necesitamos una secuencia de índices de palabras (output_mode='int'),
# no un vector tf_idf. Cada reseña se rellena o recorta a 100 palabras (mediana: 57, máximo: 125).
encoder_rnn = TextVectorization(output_mode="int", output_sequence_length=100)
encoder_rnn.adapt(train_df["text"])

texto_original = train_df["text"][1]
texto_vectorizado = encoder_rnn(tf.convert_to_tensor([texto_original]))
print(f"Texto original: {texto_original}")
print(f"Texto vectorizado: {texto_vectorizado}")

Texto original: Les cuento como me fue. Compre esta tablet a mediados de año y llego en cuatro dias a mi casa en Guadalajara, sin ningun problema con el paquete. En mi opinion, la resolucion me resulto ser deficiente para ser honesto. La uso sobre todo por las tardes para revisar correos y ver videos mientras ceno. La pantalla tactil termino siendo muy buena en el uso real.
Texto vectorizado: [[  80  711   73   10  184   61  111 1855    7   69    3  158    9   43
     4   50   30    7   19   23    4  409   40 1683  154   16    5   94
     4   19 1032    2 1289   10  201  110 1017   12  110  436    2   20
   151   51   13   35  166   12  741 1716    9  481 1053  373 8252    2
   263 1468  516  212  172  492    4    5   20  379    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]]


Ahora, entrenaremos una red neuronal con una capa de embedding, una capa SimpleRNN de 64 neuronas y una capa oculta densa de 32 neuronas con función de activación ReLU. Asimismo, entrenamos un embedding con una dimensión de salida de 64. 


In [16]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(), dtype=tf.string),
        encoder_rnn,
        tf.keras.layers.Embedding(
            input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
        ),
        tf.keras.layers.SimpleRNN(64),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3),
    ]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 64)        │     1,217,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,228,099 (4.68 MB)

 Trainable params: 1,228,099 (4.68 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [18]:
entrenar_modelo_con_parada_temprana(model, train_dataset, test_dataset)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.6345 - loss: 0.6477 - val_accuracy: 0.7484 - val_loss: 0.4654
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.8467 - loss: 0.3328 - val_accuracy: 0.7562 - val_loss: 0.4805
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9364 - loss: 0.1585 - val_accuracy: 0.7484 - val_loss: 0.6158
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9664 - loss: 0.0890 - val_accuracy: 0.7557 - val_loss: 0.6765
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9766 - loss: 0.0626 - val_accuracy: 0.7592 - val_loss: 0.7458
Epoch 6/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9894 - loss: 0.0315 - val_accuracy: 0.7592 - val_loss: 0.9584
Epoch 7/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9850 - loss: 0.0416 - val_accuracy: 0.7538 - val_loss: 1.0518
Epoch 8/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9858 - loss: 0.0382 - val_accu

### 2.1 - Probando diferentes combinaciones de capas

A continuación, probaremos 5 combinaciones distintas de capas para determinar cual tiene mejor desempeño en validación.

Para no repetir código, definimos una función que arma la red con la misma estructura de la celda anterior
(entrada de texto → `encoder_rnn` → `Embedding` → **capas recurrentes** → `Dense(32, relu)` → `Dense(3)`)
y solo cambia el bloque de capas recurrentes. Cada modelo tiene su propia capa de embedding, de modo que
ninguno comparte pesos entrenables con otro; `encoder_rnn` sí se comparte, pero no tiene pesos entrenables.

In [19]:
def construir_modelo_rnn(capas_recurrentes, nombre):
    """Arma y compila una red recurrente cuyo bloque central son las capas recibidas."""
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(), dtype=tf.string),
            encoder_rnn,
            tf.keras.layers.Embedding(
                input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
            ),
            *capas_recurrentes,
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(3),
        ],
        name=nombre,
    )
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=["accuracy"],
    )
    return model


# Guardamos el historial de entrenamiento de cada combinación para compararlas al final
historiales = {}

#### Modelo 1 - SimpleRNN bidireccional

La misma `SimpleRNN(64)` del modelo base, pero envuelta en `Bidirectional`: una copia lee la reseña de
izquierda a derecha y otra de derecha a izquierda, y sus salidas se concatenan. Así, la representación final
no depende solo de las últimas palabras de la reseña.

In [20]:
model_rnn_1 = construir_modelo_rnn(
    [tf.keras.layers.Bidirectional(tf.keras.layers.SimpleRNN(64))],
    nombre="simplernn_bidireccional",
)
model_rnn_1.summary()

historiales["1 - SimpleRNN bidireccional"] = entrenar_modelo_con_parada_temprana(
    model_rnn_1, train_dataset, test_dataset
)

Model: "simplernn_bidireccional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 100, 64)        │     1,217,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,238,403 (4.72 MB)

 Trainable params: 1,238,403 (4.72 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.5992 - loss: 0.7223 - val_accuracy: 0.7567 - val_loss: 0.4553
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.8380 - loss: 0.3493 - val_accuracy: 0.7723 - val_loss: 0.4410
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9291 - loss: 0.1715 - val_accuracy: 0.7543 - val_loss: 0.5450
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9663 - loss: 0.0917 - val_accuracy: 0.7640 - val_loss: 0.6845
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9807 - loss: 0.0558 - val_accuracy: 0.7592 - val_loss: 0.8141
Mejor época: 2 (val_accuracy = 0.7723)


#### Modelo 2 - Dos SimpleRNN apiladas

Dos capas `SimpleRNN` apiladas (64 y 32 neuronas). La primera usa `return_sequences=True` para entregarle a la
segunda la salida de cada paso de tiempo, no solo la del último.

In [21]:
model_rnn_2 = construir_modelo_rnn(
    [
        tf.keras.layers.SimpleRNN(64, return_sequences=True),
        tf.keras.layers.SimpleRNN(32),
    ],
    nombre="simplernn_apiladas",
)
model_rnn_2.summary()

historiales["2 - SimpleRNN apiladas"] = entrenar_modelo_con_parada_temprana(
    model_rnn_2, train_dataset, test_dataset
)

Model: "simplernn_apiladas"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 100, 64)        │     1,217,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 100, 64)        │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,230,179 (4.69 MB)

 Trainable params: 1,230,179 (4.69 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.5331 - loss: 0.8249 - val_accuracy: 0.6913 - val_loss: 0.5639
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.8122 - loss: 0.3858 - val_accuracy: 0.7543 - val_loss: 0.4663
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.9192 - loss: 0.1953 - val_accuracy: 0.7548 - val_loss: 0.5631
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9604 - loss: 0.1076 - val_accuracy: 0.7250 - val_loss: 0.7088
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9732 - loss: 0.0717 - val_accuracy: 0.7596 - val_loss: 0.7414
Epoch 6/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.9805 - loss: 0.0515 - val_accuracy: 0.7548 - val_loss: 0.8418
Epoch 7/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9869 - loss: 0.0401 - val_accuracy: 0.7372 - val_loss: 0.9805
Epoch 8/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9826 - loss: 0.0479 - 

#### Modelo 3 - LSTM

Reemplazamos la `SimpleRNN` por una `LSTM(64)`. Sus compuertas de entrada, olvido y salida le permiten conservar
información a lo largo de secuencias largas y mitigan el desvanecimiento del gradiente de la RNN simple.

In [22]:
model_rnn_3 = construir_modelo_rnn(
    [tf.keras.layers.LSTM(64)],
    nombre="lstm",
)
model_rnn_3.summary()

historiales["3 - LSTM"] = entrenar_modelo_con_parada_temprana(
    model_rnn_3, train_dataset, test_dataset
)

Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (None, 100, 64)        │     1,217,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,252,867 (4.78 MB)

 Trainable params: 1,252,867 (4.78 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 15s 24ms/step - accuracy: 0.6942 - loss: 0.5497 - val_accuracy: 0.7899 - val_loss: 0.3730
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.8529 - loss: 0.2853 - val_accuracy: 0.7894 - val_loss: 0.3529
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.8903 - loss: 0.2167 - val_accuracy: 0.7987 - val_loss: 0.3821
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9130 - loss: 0.1720 - val_accuracy: 0.7855 - val_loss: 0.4687
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.9336 - loss: 0.1353 - val_accuracy: 0.7924 - val_loss: 0.5210
Epoch 6/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.9486 - loss: 0.1105 - val_accuracy: 0.7943 - val_loss: 0.6850
Mejor época: 3 (val_accuracy = 0.7987)


#### Modelo 4 - GRU bidireccional con dropout

Una `GRU(64)` bidireccional (similar a la LSTM pero con menos parámetros) seguida de un `Dropout(0.5)` para
regularizar, ya que con alrededor de 18.000 reseñas de entrenamiento los modelos se sobreajustan con facilidad.

In [23]:
model_rnn_4 = construir_modelo_rnn(
    [
        tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64)),
        tf.keras.layers.Dropout(0.5),
    ],
    nombre="gru_bidireccional_dropout",
)
model_rnn_4.summary()

historiales["4 - GRU bidireccional + dropout"] = entrenar_modelo_con_parada_temprana(
    model_rnn_4, train_dataset, test_dataset
)

Model: "gru_bidireccional_dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_4 (Embedding)         │ (None, 100, 64)        │     1,217,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        49,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,271,811 (4.85 MB)

 Trainable params: 1,271,811 (4.85 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.6521 - loss: 0.6033 - val_accuracy: 0.7860 - val_loss: 0.3993
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 28s 48ms/step - accuracy: 0.8453 - loss: 0.3117 - val_accuracy: 0.7924 - val_loss: 0.3760
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 27s 46ms/step - accuracy: 0.8938 - loss: 0.2262 - val_accuracy: 0.8017 - val_loss: 0.3936
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 26s 46ms/step - accuracy: 0.9267 - loss: 0.1676 - val_accuracy: 0.8109 - val_loss: 0.4254
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 27s 46ms/step - accuracy: 0.9471 - loss: 0.1229 - val_accuracy: 0.8075 - val_loss: 0.5002
Epoch 6/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 26s 46ms/step - accuracy: 0.9620 - loss: 0.0919 - val_accuracy: 0.8158 - val_loss: 0.5521
Epoch 7/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 26s 45ms/step - accuracy: 0.9720 - loss: 0.0687 - val_accuracy: 0.8163 - val_loss: 0.6182
Epoch 8/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 26s 46ms/step - accuracy: 0.9767 - loss: 0.0587 - 

#### Modelo 5 - Dos LSTM bidireccionales apiladas con dropout

La combinación más profunda: dos `LSTM` bidireccionales apiladas (64 y 32 neuronas), con `Dropout(0.3)` entre ellas
y `Dropout(0.5)` antes de las capas densas.

In [24]:
model_rnn_5 = construir_modelo_rnn(
    [
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
        tf.keras.layers.Dropout(0.5),
    ],
    nombre="lstm_bidireccionales_apiladas",
)
model_rnn_5.summary()

historiales["5 - LSTM bidireccionales apiladas"] = entrenar_modelo_con_parada_temprana(
    model_rnn_5, train_dataset, test_dataset
)

Model: "lstm_bidireccionales_apiladas"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_5 (Embedding)         │ (None, 100, 64)        │     1,217,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 100, 128)       │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,327,107 (5.06 MB)

 Trainable params: 1,327,107 (5.06 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 47s 77ms/step - accuracy: 0.5944 - loss: 0.6291 - val_accuracy: 0.6253 - val_loss: 0.5541
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 48s 83ms/step - accuracy: 0.6329 - loss: 0.5220 - val_accuracy: 0.6287 - val_loss: 0.5422
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 47s 82ms/step - accuracy: 0.7828 - loss: 0.4080 - val_accuracy: 0.7782 - val_loss: 0.3858
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 47s 81ms/step - accuracy: 0.8710 - loss: 0.2707 - val_accuracy: 0.7929 - val_loss: 0.4068
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 47s 82ms/step - accuracy: 0.9042 - loss: 0.1986 - val_accuracy: 0.7978 - val_loss: 0.4181
Epoch 6/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 47s 81ms/step - accuracy: 0.9252 - loss: 0.1605 - val_accuracy: 0.8007 - val_loss: 0.4722
Epoch 7/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 47s 82ms/step - accuracy: 0.9365 - loss: 0.1359 - val_accuracy: 0.8031 - val_loss: 0.5851
Epoch 8/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 48s 83ms/step - accuracy: 0.9477 - loss: 0.1115 - 

#### Creación de archivos submission



In [25]:
resumen_rnn = pd.DataFrame(
    [
        {
            "modelo": nombre,
            "mejor_epoca": int(pd.Series(h.history["val_accuracy"]).idxmax()) + 1,
            "val_accuracy": max(h.history["val_accuracy"]),
            "train_accuracy": h.history["accuracy"][
                int(pd.Series(h.history["val_accuracy"]).idxmax())
            ],
        }
        for nombre, h in historiales.items()
    ]
).sort_values("val_accuracy", ascending=False)

resumen_rnn

,modelo,mejor_epoca,val_accuracy,train_accuracy
3,4 - GRU bidireccional + dropout,7,0.816317,0.972040
4,5 - LSTM bidireccionales apiladas,8,0.808012,0.947717
2,3 - LSTM,3,0.798730,0.890331
0,1 - SimpleRNN bidireccional,2,0.772350,0.837993
1,2 - SimpleRNN apiladas,8,0.765510,0.982572


In [26]:
def crear_archivo_predicciones(model, df, archivo_salida="./datos/submission.csv"):
    """Predice la etiqueta de cada reseña de `df` y guarda un CSV con el formato de
    ./datos/sample_submission.csv: columnas `id` y `answer` ('negativo', 'neutral' o 'positivo')."""
    textos = tf.data.Dataset.from_tensor_slices(df["text"].values).batch(32)

    # El modelo devuelve logits (una fila de 3 valores por reseña): la clase predicha es la de
    # mayor logit, y la traducimos de vuelta a texto con la misma lista usada al codificarlas.
    logits = model.predict(textos)
    clases = logits.argmax(axis=1)

    submission = pd.DataFrame(
        {"id": df["id"].values, "answer": [etiquetas[c] for c in clases]}
    )
    submission.to_csv(archivo_salida, index=False)

    print(f"Archivo guardado en {archivo_salida} ({len(submission)} predicciones)")
    print(submission["answer"].value_counts(normalize=True))
    return submission

In [27]:
# Cambiar por el modelo con mejor val_accuracy
modelos_rnn = [model_rnn_1, model_rnn_2, model_rnn_3, model_rnn_4, model_rnn_5]
nombres_modelo = ["Simple RNN", "Simple RNNs apiladas","LSTM","GRU","LSTM apiladas"]
modelos_rnn = {"nombre": nombres_modelo, "modelo": modelos_rnn}

for nombre, modelo in zip(modelos_rnn["nombre"], modelos_rnn["modelo"]):
    submission = crear_archivo_predicciones(modelo, dataset_eval, f"./datos/submission_{nombre}.csv")
    submission.head()

212/212 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Archivo guardado en ./datos/submission_Simple RNN.csv (6781 predicciones)
answer
negativo    0.400826
positivo    0.397876
neutral     0.201298
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Archivo guardado en ./datos/submission_Simple RNNs apiladas.csv (6781 predicciones)
answer
positivo    0.464976
negativo    0.326353
neutral     0.208671
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Archivo guardado en ./datos/submission_LSTM.csv (6781 predicciones)
answer
negativo    0.432532
positivo    0.374723
neutral     0.192744
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
Archivo guardado en ./datos/submission_GRU.csv (6781 predicciones)
answer
positivo    0.439463
negativo    0.368087
neutral     0.192449
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step
Archivo guardado en ./datos/submission_LSTM apiladas.csv (6781 predicciones)
answer
posit

## 4 - Transformers